# Word2vec example

In [4]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
import torch
from collections import Counter
import os
import numpy as np
import os
from google.colab import drive
import urllib.request




## Downloading the training text

In [5]:

developerMode=False

if developerMode:
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/Colab Notebooks/Sofie')
else:
    os.makedirs('./data', exist_ok=True)
    FILE_PATH = "data/AI2.txt"
    if not os.path.exists(FILE_PATH):
      url = "https://dl.dropboxusercontent.com/scl/fi/gkpg3zoacp55fti5gbaoj/AI2.txt?rlkey=m41ww22kul14w4fbfo9wmbk9o&dl=1"
      urllib.request.urlretrieve(url, FILE_PATH)
      print("✅ Downloaded", FILE_PATH)
    else:
      print(FILE_PATH, "Already exists")

    FILE_PATH= "data/the-verdict.txt"
    if not os.path.exists(FILE_PATH):
      url = "https://dl.dropboxusercontent.com/scl/fi/tzv0o427pwqwb4m3oap33/the-verdict.txt?rlkey=l17dke8qj8wq9u4scfbuxrbm2&dl=1"
      urllib.request.urlretrieve(url, FILE_PATH)
      print("✅ Downloaded", FILE_PATH)
    else:
      print(FILE_PATH, "Already exists")



    !ls


✅ Downloaded data/AI2.txt
✅ Downloaded data/the-verdict.txt
data  sample_data


## Reading the training text

In [6]:


file_path = "data/AI2.txt"
print("Reading the local file", file_path)
with open(file_path, "r", encoding="utf-8") as file:
    text_data = file.read()

corpus = text_data.split(".")[:500] ## taking only a part of the text sentences

Reading the local file data/AI2.txt


In [7]:
corpus[:10]

['\n\nArtificial Intelligence: Understanding a Transformative Technology\n\nArtificial Intelligence, commonly referred to as AI, is one of the most influential technological developments of the 21st century',
 ' Often surrounded by fascination, fear, and misunderstanding, AI is neither magic nor science fiction',
 ' It is a set of scientific methods, mathematical models, and computer systems designed to enable machines to perform tasks that normally require human intelligence',
 ' These tasks include learning from experience, recognizing patterns, understanding language, making decisions, and solving complex problems',
 '\n\nWhat Is Artificial Intelligence?\n\nAt its core, Artificial Intelligence is the ability of a machine to simulate aspects of human intelligence',
 ' This does not mean that machines think or feel like humans, but rather that they can process information, adapt to new data, and act in ways that appear intelligent',
 ' AI systems are typically trained on large amounts

In [8]:
# ============================================================
# 2. Preprocessing
# ============================================================
def tokenize(corpus):
    return [sentence.lower().split() for sentence in corpus]

def build_vocab(tokens):
    counts = Counter(word for sent in tokens for word in sent)
    vocab = {word: i for i, word in enumerate(counts)}
    ivocab = {i: word for word, i in vocab.items()}
    return vocab, ivocab, counts

tokens = tokenize(corpus)
vocab, ivocab, word_counts = build_vocab(tokens)
vocab_size = len(vocab)

print("Vocab size=", vocab_size)
print(list(vocab.items())[:20])


Vocab size= 1296
[('artificial', 0), ('intelligence:', 1), ('understanding', 2), ('a', 3), ('transformative', 4), ('technology', 5), ('intelligence,', 6), ('commonly', 7), ('referred', 8), ('to', 9), ('as', 10), ('ai,', 11), ('is', 12), ('one', 13), ('of', 14), ('the', 15), ('most', 16), ('influential', 17), ('technological', 18), ('developments', 19)]


In [9]:


# ============================================================
# 3. Generate skip‑gram pairs
# ============================================================
def generate_skipgram_pairs(tokens, vocab, window_size=2):
    pairs = []
    for sentence in tokens:
        encoded = [vocab[w] for w in sentence]
        for i, center in enumerate(encoded):
            for j in range(max(0, i - window_size),
                           min(len(encoded), i + window_size + 1)):
                if i != j:
                    pairs.append((center, encoded[j]))
    return pairs






## Negative sampling distribution
* Use of power < 1
* Reduces dominance of very frequent words
* Still favors common words (important for language structure)
* Gives rare words a better chance to appear

In [10]:
# ============================================================
# 4. Negative sampling distribution
# ============================================================
def negative_sampling_dist(word_counts, vocab, power=0.75):
    freqs = torch.tensor([word_counts[word] for word in vocab])
    probs = freqs.float() ** power
    return probs / probs.sum()

neg_dist = negative_sampling_dist(word_counts, vocab)

# Word2Vec model

In [11]:

# ============================================================
# 5. Word2Vec Skip‑Gram model
# ============================================================
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, embed_dim)
        self.out_embed = nn.Embedding(vocab_size, embed_dim)

    def forward(self, center, context, negatives):
        # center:   (B,)
        # context:  (B,)
        # negatives:(B, K)

        v_center = self.in_embed(center)               # (B, D)
        v_context = self.out_embed(context)             # (B, D)
        v_neg = self.out_embed(negatives)               # (B, K, D)

        pos_score = torch.sum(v_center * v_context, dim=1)
        pos_loss = torch.log(torch.sigmoid(pos_score))

        neg_score = torch.bmm(v_neg, v_center.unsqueeze(2)).squeeze(-1) ## batch matrix multiplication
        neg_loss = torch.log(torch.sigmoid(-neg_score)).sum(dim=1)

        return -(pos_loss + neg_loss).mean()

In [ ]:


if True:
  pairs = generate_skipgram_pairs(tokens, vocab, window_size=3)

  # ============================================================
  # 6. Training
  # ============================================================
  EMBED_DIM = 50
  NEG_SAMPLES = 5
  EPOCHS = 200
  LR = 0.001

  model = Word2Vec(vocab_size, EMBED_DIM)
  optimizer = optim.Adam(model.parameters(), lr=LR)

  for epoch in range(EPOCHS):
      total_loss = 0.0
      random.shuffle(pairs)

      for center, context in pairs:
          negatives = torch.multinomial(
              neg_dist, NEG_SAMPLES, replacement=True
          )

          center = torch.tensor([center])
          context = torch.tensor([context])
          negatives = negatives.unsqueeze(0)

          if random.random() < 0.00001:
            print("center=",list(vocab.items())[center.item()])
            print("context=",[list(vocab.items())[n]  for n in [context]])
            print("negatives=", [list(vocab.items())[n]  for n in negatives.tolist()[0]])

          loss = model(center, context, negatives)

          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

          total_loss += loss.item()


      print(f"Epoch {epoch:03d} | Loss: {total_loss:.4f}")

  embeddings = model.in_embed.weight.detach()
  np.savez(file="embeddings.npz",embeddings=embeddings)
else:
  embeddings = torch.tensor(np.load("embeddings.npz")["embeddings"])



Epoch 000 | Loss: 313904.0402
Epoch 001 | Loss: 210441.4985
Epoch 002 | Loss: 151142.7567
Epoch 003 | Loss: 113493.4078
Epoch 004 | Loss: 87873.3741
Epoch 005 | Loss: 70954.1437
Epoch 006 | Loss: 59760.2892
Epoch 007 | Loss: 51629.8501
Epoch 008 | Loss: 45879.8177
center= ('lies', 568)
context= [('foundation', 809)]
negatives= [('together,', 413), ('probabilistically,', 1064), ('correct', 765), ('task', 144), ('literacy', 951)]
Epoch 009 | Loss: 41764.6356
Epoch 010 | Loss: 38316.2196
Epoch 011 | Loss: 36154.1834
center= ('from', 56)
context= [('competence;', 1228)]
negatives= [('transforms', 961), ('humans', 411), ('operations', 1047), ('modeling', 716), ('are', 98)]
Epoch 012 | Loss: 33990.4636
Epoch 013 | Loss: 32701.0968
Epoch 014 | Loss: 31802.5627
Epoch 015 | Loss: 30748.6097
center= ('error,', 293)
context= [('and', 27)]
negatives= [('eigenvalues,', 1050), ('model’s', 539), ('and', 27), ('principle', 210), ('neither', 30)]
Epoch 016 | Loss: 29765.6025
Epoch 017 | Loss: 29201.088

In [ ]:
# ============================================================
# 7. Inspect embeddings
# ============================================================

def cosine_similarity(word1, word2):
    v1 = embeddings[vocab[word1]]
    v2 = embeddings[vocab[word2]]
    return F.cosine_similarity(v1, v2, dim=0).item()

print("\nCosine similarities:")
print("artificial vs intelligence:", cosine_similarity("artificial", "intelligence"))
print("computer vs economy:", cosine_similarity("computer", "economy"))
print("machine vs intelligence:", cosine_similarity("machine", "intelligence"))
print("computer vs machine:", cosine_similarity("computer", "machine"))



In [2]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE



ivocab = {i: word for word, i in vocab.items()}

#embeddings = model.in_embed.weight.detach()
words =[ 'artificial', 'intelligence:', 'computer', 'machine', 'learning',
        'behavior','economy','optimization','loss','error', 'structure','job','power']
X = embeddings.numpy()
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)
plt.figure(figsize=(8, 6))
#plt.figure(figsize=(.show()
plt.scatter(X_2d[:len(words), 0], X_2d[:len(words), 1], s=40)

for i, word in enumerate(words):
    plt.annotate(
        word,
        (X_2d[i, 0], X_2d[i, 1]),
        fontsize=10,
        alpha=0.8
    )

plt.title("Word2Vec embeddings (PCA projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.tight_layout()


NameError: name 'vocab' is not defined

In [ ]:
words

['artificial',
 'intelligence:',
 'understanding',
 'a',
 'transformative',
 'technology',
 'intelligence,',
 'commonly',
 'referred',
 'to',
 'as',
 'ai,',
 'is',
 'one',
 'of',
 'the',
 'most',
 'influential',
 'technological',
 'developments']

In [ ]:
vocab

{'artificial': 0,
 'intelligence:': 1,
 'understanding': 2,
 'a': 3,
 'transformative': 4,
 'technology': 5,
 'intelligence,': 6,
 'commonly': 7,
 'referred': 8,
 'to': 9,
 'as': 10,
 'ai,': 11,
 'is': 12,
 'one': 13,
 'of': 14,
 'the': 15,
 'most': 16,
 'influential': 17,
 'technological': 18,
 'developments': 19,
 '21st': 20,
 'century': 21,
 'often': 22,
 'surrounded': 23,
 'by': 24,
 'fascination,': 25,
 'fear,': 26,
 'and': 27,
 'misunderstanding,': 28,
 'ai': 29,
 'neither': 30,
 'magic': 31,
 'nor': 32,
 'science': 33,
 'fiction': 34,
 'it': 35,
 'set': 36,
 'scientific': 37,
 'methods,': 38,
 'mathematical': 39,
 'models,': 40,
 'computer': 41,
 'systems': 42,
 'designed': 43,
 'enable': 44,
 'machines': 45,
 'perform': 46,
 'tasks': 47,
 'that': 48,
 'normally': 49,
 'require': 50,
 'human': 51,
 'intelligence': 52,
 'these': 53,
 'include': 54,
 'learning': 55,
 'from': 56,
 'experience,': 57,
 'recognizing': 58,
 'patterns,': 59,
 'language,': 60,
 'making': 61,
 'decisions,